## **IMPORTING THE NECESSARY LIBRARY.**

In [1]:
import pandas as pd

**1. Reading in the data.**

In [ ]:
Edata = pd.read_excel("../cleaned_data/education_level_cleaned.xlsx", sheet_name="education_level")
Cdata = pd.read_excel("../cleaned_data/cis_data_cleaned.xlsx", sheet_name="cis_data")

**2. Standarizing the column names.**

In [3]:
Edata.columns = Edata.columns.str.strip().str.lower()
Cdata.columns = Cdata.columns.str.strip().str.lower()

**3. Dropping Unnecessary Columns.**

- Taking out the geography column because we are working on the same location **"Ontario"**, so keeping the column there makes it redundant.
- Removing the **"wage rate"** column which is coming from the Edata because its redundant.
- Removing the **"All Employees"** category from the type of work column because its a duplicate.
- Removing th **"Age_group"** column because the age categories in the datasets dont merge.

In [4]:
for col in ["geography"]:
    if col in Edata.columns:
        Edata = Edata.drop(columns=[col])
    if col in Cdata.columns:
        Cdata = Cdata.drop(columns=[col])

if "wage rate" in Edata.columns:
    Edata = Edata.drop(columns=["wage rate"])

if "type of work" in Edata.columns:
    Edata = Edata[~Edata["type of work"].str.lower().str.contains("all")]

for col in ["age group", "age_group"]:
  if col in Edata.columns:
    Edata = Edata.drop(columns=[col])
  if col in Cdata.columns:
    Cdata = Cdata.drop(columns=[col])

**4. Aggregating the Cdata.**

In [5]:
Cdata_grouped = (
    Cdata.groupby(["year", "immigrant_status", "education", "gender"])
         .agg({
             "earnings": "mean",
             "wages_salary": "mean",
             "total_income": "mean",
             "person_id": "count"
         })
         .reset_index()
)

**5. Standardizing the immigrant category.**

In [6]:
immigrant_map = {
    "Born in Canada (Non-Immigrant)": "Born in Canada (non-immigrant)",
    "Landed immigrant": "Immigrant"
}

Cdata_grouped["immigrant_status"] = (
    Cdata_grouped["immigrant_status"].map(immigrant_map)
)

**6. Merging the Datasets.**

In [7]:
merged = pd.merge(
    Edata,
    Cdata_grouped,
    left_on=["year", "immigrant", "education"],
    right_on=["year", "immigrant_status", "education"],
    how="inner"
)

**7. Dropping the columns not needed in the merged dataset.**

In [8]:
cols_to_drop = ["_id", "household_id", "person_id", "immigrant_status"]
merged_clean = merged.drop(columns=[c for c in cols_to_drop if c in merged.columns])

**8. Saving the new merged dataset.**

In [9]:
merged_clean.to_excel("merged_output_cleaned.xlsx", index=False)